In [ ]:
# import numpy as np
# import os

# # Load the .npy files
# # X_train = np.load( "task1_X_train.npy")
# y_train = np.load( "/home/sombit-ng-nonadm/vaibhav_intern_pg/CS230Project/task2_damage_state_1/task2_y_train.npy")
# X_test = np.load( "/home/sombit-ng-nonadm/vaibhav_intern_pg/CS230Project/task2_damage_state_1/task2_X_test.npy")
# y_test = np.load("/home/sombit-ng-nonadm/vaibhav_intern_pg/CS230Project/task2_damage_state_1/task2_y_test.npy")

# # Print shapes of the arrays
# # print(f"X_train shape: {X_train.shape}")
# print(f"y_train shape: {y_train.shape}")
# print(f"X_test shape: {X_test.shape}")
# print(f"y_test shape: {y_test.shape}")


y_train shape: (11811, 2)
X_test shape: (1460, 224, 224, 3)
y_test shape: (1460, 2)


In [ ]:
# import numpy as np

# file_path = "/home/sombit-ng-nonadm/vaibhav_intern_pg/CS230Project/task2_damage_state_1/task2_X_train.npy"

# X_train = np.load(file_path, allow_pickle=True)  # Allow pickle loading
# print(f"X_train shape: {X_train.shape}")




X_train shape: (11811, 224, 224, 3)


In [1]:
import torch

device = torch.device("cpu")  # Force using CPU
# model.to(device)  # Move model to CPU

# When running inference or training, ensure tensors are on CPU



In [1]:
import numpy as np

X_labeled = np.load("/home/sombit-ng-nonadm/vaibhav_intern_pg/BTP+AIproject/Task_1_scene_level/X_labeled_small.npy", mmap_mode='r')
y_labeled = np.load("/home/sombit-ng-nonadm/vaibhav_intern_pg/BTP+AIproject/Task_1_scene_level/y_labeled_small.npy", mmap_mode='r')
# X_unlabeled = np.load("/home/sombit-ng-nonadm/vaibhav_intern_pg/CS230Project/Task_1_scene_level/X_unlabeled.npy", mmap_mode='r')
print(f" {y_labeled.shape}")
# print(f" {X_unlabeled.shape}")
print(f"{X_labeled.shape}")

 (4051, 3)
(4051, 224, 224, 3)


In [3]:

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
import pandas as pd



2025-04-09 16:05:03.163787: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-09 16:05:03.287745: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744214703.320184  185525 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744214703.327835  185525 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1744214703.351400  185525 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [3]:
from sklearn.model_selection import train_test_split

def split_data(X, y, splitsize=0.2, shuffle=True, stratify=False, seed=42):
    if stratify:
        X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=splitsize, stratify=y, random_state=seed)
    else:
        X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=splitsize, random_state=seed)

    return X_train, X_val, y_train, y_val



ProcessData

In [12]:


def image_generators(train_dir, val_dir, X_train_df, X_val_df, batch_size, image_height, image_width):
    """
    Creates image data generators for training and validation.

    Args:
        train_dir (str): Path to training image directory.
        val_dir (str): Path to validation image directory.
        X_train_df (pd.DataFrame): DataFrame with 'image' column (filenames) and 'label' column (classes).
        X_val_df (pd.DataFrame): DataFrame with 'image' column (filenames) and 'label' column (classes).
        batch_size (int): Batch size for training.
        image_height (int): Target image height.
        image_width (int): Target image width.

    Returns:
        train_generator, val_generator
    """

    # Data augmentation for training images
    train_datagen = ImageDataGenerator(
        rotation_range=10,
        width_shift_range=0.1,
        height_shift_range=0.1,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        rescale=1./255
    )

    # Only rescale validation images
    val_datagen = ImageDataGenerator(rescale=1./255)

    # Training generator
    train_generator = train_datagen.flow_from_dataframe(
        dataframe=X_train_df,
        directory=train_dir,  # Path to actual image folder
        x_col='image',        # Column with filenames
        y_col='label',        # Column with class labels
        target_size=(image_height, image_width),
        batch_size=batch_size,
        class_mode='categorical'  # Use 'categorical' for multi-class classification
    )

    # Validation generator
    val_generator = val_datagen.flow_from_dataframe(
        dataframe=X_val_df,
        directory=val_dir,  # Path to actual image folder
        x_col='image',
        y_col='label',
        target_size=(image_height, image_width),
        batch_size=batch_size,
        class_mode='categorical'
    )

    return train_generator, val_generator


In [5]:
X_train, X_val, y_train, y_val= split_data(X_labeled, y_labeled, 0.2, True, True, 42)

In [14]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [7]:
import os
import cv2
import pandas as pd

output_folder = "X_labeled_train"
os.makedirs(output_folder, exist_ok=True)

# Fix pixel values by shifting and scaling to 0-255
X_train_scaled = X_train - X_train.min()  # Shift to 0
X_train_scaled = X_train_scaled / X_train_scaled.max()  # Scale to 0-1
X_train_scaled = (X_train_scaled * 255).astype(np.uint8)  # Convert to 0-255

# Save images and store file paths
image_paths = []  # Initialize list to store image paths
for i, img_array in enumerate(X_train_scaled):
    image_path = os.path.join(output_folder, f"image_{i}.jpg")

    cv2.imwrite(image_path, img_array)  # Save image
    image_paths.append(image_path)  # Append path to list ✅
labels = np.argmax(y_train, axis=1)

# Create DataFrame
X_labelled_train_df = pd.DataFrame({"image": image_paths, "label": labels})



In [8]:

from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, Model
from tensorflow.keras.layers import Dropout, GlobalMaxPooling2D, Dense

def build_model(num_classes):
    """
    Loads VGG16 with pre-trained ImageNet weights and modifies the classifier.
    
    Args:
        num_classes (int): Number of output classes.

    Returns:
        model (tensorflow.keras.Model): Modified VGG16 model.
    """

    # Load VGG16 base model (without the fully connected top)
    vgg = VGG16(include_top=False, weights='imagenet', input_shape=(224, 224, 3))

    # Freeze convolutional layers
    for layer in vgg.layers:
        layer.trainable = False

    # Add custom classification head
    x = GlobalMaxPooling2D()(vgg.output)  
    x = Dense(512, activation='relu')(x)  
    x = Dropout(0.5)(x)
    x = Dense(num_classes, activation='softmax')(x)  

    # Create final model
    model = Model(inputs=vgg.input, outputs=x)

    return model

# Example usage
if __name__ == "__main__":
    model = build_model(num_classes=3) 
    model.summary()  # Print model architecture


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling2d            │ (None, 512)            │             0 │
│ (GlobalMaxPooling2D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │         1,539 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,978,883 (57.14 MB)

 Trainable params: 264,195 (1.01 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # Disable GPU

import tensorflow as tf
print("Running on CPU:", tf.config.list_physical_devices('GPU'))


2025-04-09 16:13:11.418078: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-09 16:13:11.439837: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744215191.455618  185927 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744215191.460397  185927 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1744215191.473057  185927 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Running on CPU: []


2025-04-09 16:13:14.572734: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2025-04-09 16:13:14.572764: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:167] env: CUDA_VISIBLE_DEVICES="-1"
2025-04-09 16:13:14.572770: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:170] CUDA_VISIBLE_DEVICES is set to -1 - this hides all GPUs from CUDA
2025-04-09 16:13:14.572774: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:178] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
2025-04-09 16:13:14.572778: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:183] retrieving CUDA diagnostic information for host: hela
2025-04-09 16:13:14.572779: I external/local_xla/xla/stream_executor/cuda/cuda_dia

In [9]:
# val data set
output_folder = "X_labeled_val"
os.makedirs(output_folder, exist_ok=True)

# Fix pixel values by shifting and scaling to 0-255
X_val_scaled = X_val - X_val.min()  # Shift to 0
X_val_scaled = X_val_scaled / X_val_scaled.max()  # Scale to 0-1
X_val_scaled = (X_val_scaled * 255).astype(np.uint8)  # Convert to 0-255

# Save images and store file paths
X_val_filenames = []  # Initialize list to store image paths
for i, img_array in enumerate(X_val_scaled):
    image_path = os.path.join(output_folder, f"image_{i}.jpg")

    cv2.imwrite(image_path, img_array)  # Save image
    X_val_filenames.append(image_path)  # Append path to list ✅
val_labels = np.argmax(y_val, axis=1)

# Create DataFrame
X_labelled_val_df = pd.DataFrame({"image": X_val_filenames, "label": val_labels})

In [15]:
# path="/home/sombit-ng-nonadm/vaibhav_intern_pg/CS230Project/task2_damage_state_1"
image_size=X_train[0].shape
batch_size = 32
img_height = image_size[1]
img_width = image_size[0]
# X_train_df = pd.DataFrame({'image': X_train, 'label': y_train})
# X_val_df = pd.DataFrame({'image': X_val, 'label': y_val})
# Convert labels (0 → "damaged", 1 → "undamaged")
X_labelled_train_df["image"] = X_labelled_train_df["image"].apply(lambda x: os.path.basename(x))

X_labelled_val_df["image"] = X_labelled_val_df["image"].apply(lambda x: os.path.basename(x))
X_labelled_train_df["label"] = X_labelled_train_df["label"].apply(lambda x: os.path.basename(str(x)))
X_labelled_val_df["label"] = X_labelled_val_df["label"].apply(lambda x: os.path.basename(str(x)))


# Example usage
train_generator, val_generator = image_generators(
    train_dir="/home/sombit-ng-nonadm/vaibhav_intern_pg/BTP+AIproject/VGG16/X_labeled_train",   # Folder where training images are stored
    val_dir="/home/sombit-ng-nonadm/vaibhav_intern_pg/BTP+AIproject/VGG16/X_labeled_val",       # Folder where validation images are stored
    X_train_df=X_labelled_train_df, 
    X_val_df=X_labelled_val_df, 
    batch_size=32, 
    image_height=image_size[1], 
    image_width=image_size[0]
)


Found 3240 validated image filenames belonging to 3 classes.
Found 811 validated image filenames belonging to 3 classes.


In [16]:
import os

missing_files = [img for img in X_labelled_train_df["image"] if not os.path.exists(os.path.join("X_labeled_train", img))]
print(f"Missing files: {len(missing_files)}")


Missing files: 0


In [17]:
print(X_labelled_train_df.head())  # Should show filenames and labels
print(X_labelled_val_df.head())    # Should show filenames and labels


         image label
0  image_0.jpg     2
1  image_1.jpg     2
2  image_2.jpg     2
3  image_3.jpg     1
4  image_4.jpg     1
         image label
0  image_0.jpg     0
1  image_1.jpg     0
2  image_2.jpg     1
3  image_3.jpg     2
4  image_4.jpg     1


In [18]:
from tensorflow.keras.optimizers import Adam

model = build_model(num_classes=3)  # Change num_classes for multi-class

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.0001), 
              loss='categorical_crossentropy', 
              metrics=['accuracy'])

# Train the model
model.fit(train_generator, validation_data=val_generator, epochs=10)


/home/sombit-ng-nonadm/miniconda3/envs/vaibhav_intern/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 79s 758ms/step - accuracy: 0.5120 - loss: 1.0253 - val_accuracy: 0.8064 - val_loss: 0.5184
Epoch 2/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 78s 764ms/step - accuracy: 0.7403 - loss: 0.6179 - val_accuracy: 0.8249 - val_loss: 0.4624
Epoch 3/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 77s 758ms/step - accuracy: 0.7530 - loss: 0.5507 - val_accuracy: 0.8360 - val_loss: 0.4319
Epoch 4/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 77s 754ms/step - accuracy: 0.7872 - loss: 0.5060 - val_accuracy: 0.8286 - val_loss: 0.4246
Epoch 5/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 77s 753ms/step - accuracy: 0.7906 - loss: 0.4757 - val_accuracy: 0.8335 - val_loss: 0.4124
Epoch 6/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 76s 744ms/step - accuracy: 0.7995 - loss: 0.4764 - val_accuracy: 0.8422 - val_loss: 0.4012
Epoch 7/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 78s 765ms/step - accuracy: 0.8226 - loss: 0.4372 - val_accuracy: 0.8348 - val_loss: 0.3966
Epoch 8/10
102/102 ━━━━━━━━━━━━━━━━━━━━ 77s 751ms/step - accuracy: 0.8191 - loss: 0

In [19]:
model.save("vgg16_modelScene_small_final.h5")  # Saves architecture, weights, and optimizer state
